In [0]:
# step 1 -> Read customers table from bronze and store in df

# Step 2 -> Display the dataframe and try to identify - what cleaning can be done

# Step 3 -> Apply the required transformation steps which you identified and then again store in dataframe


# Step 4 -> in final dataframe we will apply scd - type 1 and then store in silver schema ( before that create silver schema using SQL )

# Step 5 write a function in same notebook which clean and transfor the customers data from bronze and returns the dataframe

# step 6 -> Write a function which can do scd 1 - if we provide parameters like source dataframe, business_key and target_table [ After creating this function (scd1) we will take it to common_utils and then reuse it here and in other notebooks]

# step 7 -> Fully modularize the code ( hardcoded values to json config, proper logging setup and all the resusable function should have used)

# step 1 -> Read customers table from bronze and store in df

In [0]:
df = spark.read.table("retaildataplatform.bronze.sqlserver_customers")

# Step 2 -> Display the dataframe and try to identify - what cleaning can be done

In [0]:
df.display()

In [0]:
# customer name should small 


# customer_name -> first_name and last_name then drop customer_name


# city should be capital


# postcode should not contain .0


# drop file path


# valid_from and valid_to should be utc timestamp


# add country column with value 'USA'


# identify the business keys for this table


# Step 3 -> Apply the required transformation steps which you identified and then again store in dataframe

In [0]:
# customer name should small 
df = df.withColumn("customer_name", F.lower(df["customer_name"]))

# customer_name -> first_name and last_name then drop customer_name
df = df.withColumn("name_parts", F.split(F.col("customer_name"),","))\
    .withColumn("first_name", F.expr("get(name_parts,1)"))\
        .withColumn("last_name", F.expr("get(name_parts,0)"))\
        .drop("name_parts")\
        .drop("customer_name")

# city should be capital
df = df.withColumn("city", F.upper(df["city"]))

# postcode should not contain .0
df = df.withColumn("postcode", F.regexp_replace(F.col("postcode"), "\.0$", ""))

# drop file path
df = df.drop("file_path")

# valid_from and valid_to should be utc timestamp
df = df.withColumn("valid_from", F.from_unixtime(F.expr("try_cast(valid_from as bigint)"))) \
             .withColumn("valid_to", F.from_unixtime(F.expr("try_cast(valid_to as bigint)")))

# add country column with value 'USA'
df = df.withColumn('country',F.lit('USA'))

# identify the business keys for this table

df = df.dropDuplicates(["customer_id"])

In [0]:
df.display()

In [0]:
df.groupBy("customer_id").count().alias("count").filter("count > 1").display()
df.filter("customer_id = 34311371").display()

# Step 4 write a function in same notebook which clean and transfor the customers data from bronze and returns the dataframe

In [0]:
df = spark.read.table("retaildataplatform.bronze.sqlserver_customers")

def transform_customer(df):

    df = df.withColumn("customer_name", F.lower(df["customer_name"]))\
        .withColumn("name_parts", F.split(F.col("customer_name"),","))\
        .withColumn("first_name", F.expr("get(name_parts,1)"))\
        .withColumn("last_name", F.expr("get(name_parts,0)"))\
        .withColumn("city", F.upper(df["city"]))\
        .withColumn("postcode", F.regexp_replace(F.col("postcode"), "\.0$", ""))\
        .withColumn("valid_from", F.from_unixtime(F.expr("try_cast(valid_from as bigint)"))) \
        .withColumn("valid_to", F.from_unixtime(F.expr("try_cast(valid_to as bigint)")))\
        .withColumn('country',F.lit('USA'))

    df = df.drop("name_parts","customer_name","file_path")

    df = df.dropDuplicates(["customer_id"])

    return df.select(
        "customer_id",
        "first_name",
        "last_name",
        "tax_id",
        "tax_code",
        "country",
        "state",
        "city",
        "postcode",
        "street",
        "number",
        "unit",
        "region",
        "district",
        "lon",
        "lat",
        "ship_to_address",
        "valid_from",
        "valid_to",
        "units_purchased",
        "loyalty_segment",
        "last_update_ts"
    )

In [0]:
df = spark.read.table("retaildataplatform.bronze.sqlserver_customers")
cleaned_customer = transform_customer(df)
cleaned_customer.display()


In [0]:
df.display()

# Step 5 -> in final dataframe we will apply scd - type 1 and then store in silver schema ( before that create silver schema using SQL )

## SCD 1

> Upsert -> update + insert 

Update -> Changed Rows


Insert -> New Rows


Business Key or Unique Identifier

SCD 1 -> Compare source and target 


no silver schema
no customer table in silver 


check if table exist or not

if exist then do scd 1

otherwise do full load (write the table in delta format)



In [0]:
source_df = df

target_df = ?

In [0]:
if spark.catalog.tableExists("retaildataplatform.silver.customers"):
    print("Table Exists - Now Proceeding with SCD 1")




else:
    print("Table Does Not Exist - Now Creating Table")
    spark.sql("""Create Schema if not exists silver """)
    df.write.format("delta").mode("overwrite").saveAsTable("retaildataplatform.silver.customers")
